In [1]:
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from dotenv import load_dotenv

api_key= os.getenv("cohere_api_key")
api_key=api_key
co= cohere.Client(api_key)


In [2]:
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""

texts= text.split('.')

texts= [t.strip(' \n') for t in texts]

In [23]:
response= co.embed(
    texts=texts,
    input_type="search_document",

).embeddings

embeds=np.array(response)
print(embeds.shape)

[[0.92626953, -0.9633789, 1.6796875, -2.4238281, -0.17822266, -0.8935547, 1.5205078, -1.7080078, 1.2675781, 1.1347656, 0.80078125, 0.06689453, -1.1298828, 0.058685303, 0.96875, -1.1523438, 0.5385742, -0.2758789, 2.015625, 0.2265625, -0.038238525, 0.9814453, -0.38916016, 0.061706543, -0.6328125, 2.9160156, 0.56640625, 0.25756836, 0.45507812, 0.5649414, 0.60546875, 0.84521484, -0.32885742, 0.8154297, 0.75097656, 0.1953125, 1.2666016, -0.3347168, 2.4941406, -0.17041016, -1.6650391, 1.4970703, -2.1777344, -0.93603516, -0.46166992, -0.5600586, -0.25170898, 0.90185547, 0.07208252, -1.1113281, 1.4042969, -1.2246094, 2.8828125, 0.96191406, 0.4013672, -1.3710938, -0.5004883, -0.23400879, 2.0605469, -2.7871094, -1.0546875, -1.4423828, 1.9355469, -0.5786133, -0.4206543, 0.8066406, -0.6352539, -2.0917969, 0.76660156, 1.3798828, -5.1289062, -0.20947266, -0.32373047, -1.1972656, -0.8833008, 1.8359375, -0.13891602, -0.54296875, 0.43603516, 2.0273438, -0.38330078, -1.6845703, -0.84277344, 1.4697266, -

In [4]:
import faiss 
dim= embeds.shape[1] #4096
index= faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))


True


In [21]:
def search(query, num_of_results=3):
    query_embed= co.embed(texts=[query], input_type="search_query").embeddings[0] #.embeddings0 fetches the embedding vector for query 

    distances, similar_items_ids= index.search(np.float32([query_embed]), num_of_results)

    texts_np= np.array(texts)
    results= pd.DataFrame(data={'texts': texts_np[similar_items_ids[0]], 'distance':distances[0]})
#                                            ^^^this is fancy indexing allowing it to fetch the content on the index given
    print(f"Query: '{query}'\nNearest neighbors:")
    
    # Display full sentences without truncation
    for i, (idx, row) in enumerate(results.iterrows()):
        print(f"{i}: {row['texts']} (distance: {row['distance']:.6f})")
    
    return results

In [22]:
query= "how precise was the science?"
results=search(query)
results

Query: 'how precise was the science?'
Nearest neighbors:
0: It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics (distance: 9659.714844)
1: Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects (distance: 10615.987305)
2: Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar (distance: 10969.983398)


,texts,distance
0,It has also received praise from many astronom...,9659.714844
1,Interstellar uses extensive practical and mini...,10615.987305
2,Caltech theoretical physicist and 2017 Nobel l...,10969.983398


In [ ]:
#we compare semantic search to keyword search
#Lexical search, also known as keywords search, refers to a search algorithm based on the word-level analysis of text
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenize_doc=[]#empty array or list
    for token in text.lower().split(): #lower case then split on whitespaces
        token= token.strip(string.punctuation) # remove punc from the start and the end of the tokens

        if len(token)>0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenize_doc.append(token)
    return tokenize_doc